## Project Details


## Import Packages

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Descriptions

• Customer table used to only keep real customers. For example, test accounts are removed from the data.

• Visit Plans History table gives us the most recent visit plan for each timestamp. This tells us the customer’s anchor date and frequency.

• Orders table is used to check the orders in GA data against what was actually ordered.

• Sales table is used to see what was actually sold to customers. This is different than the orders table due to some items sometimes being sold out.

• Cutoff table - Each customer has a specific time of day they are supposed to order by. This time is generally 5pm but the Cutoff times table gives us the exceptions.

• Operating hours table gives us the current frequency, anchor day, and anchor date for each customer. The visit plans table is a historical version of this table.

• Material table gives us descriptive information for each material id. Such as size, category and brand.


## Rules & Calculations

• Anchor Date, cutoff time from each plant / sales office, and frequency are used to
compute when the customer is supposed to order next for each event in the GA data.
The most recent anchor date and frequency are retrieved from the visit plans history
table. These with the cutoff time give when the exact time and day when a customer is
expected to order by.

• The time between two expected order by times is called an “Order Window”.

• All timestamps in Google Analytics are logged in EST, but the cutoff times are local to the customer. An
adjusted timestamp column must be computed using the time zone of the sales office
that customer orders from.

## Import Data

### Cutoff Times

In [0]:
# Loading Cutoff Times Data

# Time of day customers must have their orders in by, by sales office
#     SALES_OFFICE - which sales office the customer orders from
#     PLANT_ID - which bottling plant the customer receives their order from
#     CUTOFFTIME_C - time the customer must have their order in. If NA then time is 5pm
#     SHIPPING_CONDITION_TIME - general shipping strategy for delivery of goods
#     DISTRIBUTION_MODE - mode of distrobution for a customer.

cutoff = pd.read_csv("cutoff_times.csv")

# Should have 220 rows and 5 columns

print(cutoff.shape)

In [0]:
# head
print(cutoff.head())

In [0]:
# Cutoff Times Summary Stats
print(cutoff.isna().sum())
print(cutoff.info())
print(cutoff.describe(include="all"))

In [0]:
# Converting cutoff times:
cutoff["CUTOFFTIME__C"] = cutoff["CUTOFFTIME__C"].fillna("5:00:00 PM")
cutoff["CUTOFFTIME__C"] = pd.to_datetime(cutoff["CUTOFFTIME__C"], format="%I:%M:%S %p")
cutoff["CUTOFFTIME__C"].dtypes

In [0]:
# finding unique sales offices
unique_offices = cutoff["SALES_OFFICE"].unique()
print(unique_offices)

In [0]:
tz_map = {
    # Utah ~ Mountain
    "Draper, UT": "America/Denver",
    "Logan, UT": "America/Denver",
    "Richfield, UT": "America/Denver",
    "Price, UT": "America/Denver",
    "Ogden, UT": "America/Denver",

    # Colorado ~ Mountain
    "Grand Junction, CO": "America/Denver",
    "Alamosa, CO": "America/Denver",
    "Glenwood, CO": "America/Denver",
    "Johnstown, CO": "America/Denver",
    "Denver, CO": "America/Denver",
    "Pueblo, CO": "America/Denver",
    "Colorado Springs, CO": "America/Denver",

    # Idaho ~ mostly Mountain, except Lewiston → Pacific
    "Idaho Falls, ID": "America/Denver",
    "Pocatello, ID": "America/Denver",
    "Boise, ID": "America/Denver",
    "Twin Falls, ID": "America/Denver",
    "Lewiston, ID": "America/Los_Angeles",

    # Nebraska ~ Central
    "Scottsbluff, NE": "America/Chicago",

    # Wyoming ~ Mountain
    "Cheyenne, WY": "America/Denver",

    # Arizona ~ Mountain (no DST)
    "Flagstaff, AZ": "America/Phoenix",
    "Kingman, AZ": "America/Phoenix",
    "Show Low, AZ": "America/Phoenix",
    "Chinle, AZ": "America/Phoenix",
    "Tempe, AZ": "America/Phoenix",
    "Glendale, AZ": "America/Phoenix",
    "Prescott, AZ": "America/Phoenix",
    "Cochise, AZ": "America/Phoenix",
    "Tucson, AZ": "America/Phoenix",

    # Nevada ~ Pacific
    "Reno, NV": "America/Los_Angeles",
    "Elko, NV": "America/Los_Angeles",

    # Washington ~ Pacific
    "Walla Walla, WA": "America/Los_Angeles",
    "Wenatchee, WA": "America/Los_Angeles",
    "Bellevue, WA": "America/Los_Angeles",
    "Bremerton, WA": "America/Los_Angeles",
    "South Sound, WA": "America/Los_Angeles",
    "Arlington, WA": "America/Los_Angeles",

    # Oregon ~ Pacific
    "LaGrande, OR": "America/Los_Angeles",
    "Pendleton, OR": "America/Los_Angeles",
    "Wilsonville, OR": "America/Los_Angeles",
    "Bend, OR": "America/Los_Angeles",
    "Eugene, OR": "America/Los_Angeles",

    # New Mexico ~ Mountain
    "Albuquerque, NM": "America/Denver",

    # Placeholder/fallback
    "0": "America/New_York"  # replace with correct office if known
}

In [0]:
# Converting cutoff time to UTC:
today = pd.Timestamp.today().normalize()  
cutoff["CUTOFF_DATETIME"] = cutoff["CUTOFFTIME__C"].apply(lambda t: today + pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second))

# Convert to UTC
cutoff["CUTOFFTIME_UTC"] = cutoff.apply(
    lambda row: row["CUTOFF_DATETIME"]
        .tz_localize(tz_map.get(row["SALES_OFFICE"], "America/New_York"))
        .tz_convert("UTC"),
    axis=1
)

print(cutoff[cutoff["SALES_OFFICE"] == 'Draper, UT'][["CUTOFFTIME_UTC", "CUTOFFTIME__C"]])

In [0]:
# Cutoff dummy variables:
cutoff_dummies = pd.get_dummies(cutoff, columns=['DISTRIBUTION_MODE'], drop_first=True)
print(cutoff_dummies.head())

In [0]:
# Distribution mode chart
counts = cutoff['DISTRIBUTION_MODE'].value_counts()

# Plot
counts.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Frequency of Distribution Modes")
plt.ylabel("Count")
plt.xlabel("Distribution Mode")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Time of day customers must have their orders in by, by sales office
#     SALES_OFFICE - which sales office the customer orders from
#     PLANT_ID - which bottling plant the customer receives their order from
#     CUTOFFTIME_C - time the customer must have their order in. If NA then time is 5pm
#     SHIPPING_CONDITION_TIME - general shipping strategy for delivery of goods
#     DISTRIBUTION_MODE - mode of distrobution for a customer.


In [0]:
# Distribution mode chart
counts = cutoff['SALES_OFFICE'].value_counts()

# Plot
counts.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Clients per Sales Office")
plt.ylabel("Count")
plt.xlabel("Sales Office")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Customer

In [0]:
# Loading Customer Data

# customer info
#     SALES_OFFICE - sales office they order from
#     CUSTOMER_NUMBER - uniqe identifier for each customer
#     SALES_OFFICE_DESCRIPTION - location of sales office
#     SHIPPING_CONDITIONS_DESCRIPTION - general shipping strategy for the delivery of goods
#     COLD_DRINK_CHANNEL_DESCRIPTION - higher level grouping of customers such as retail, wholesale, eating/drinking
#     CUSTOMER_SUB_TRADE_CHANNEL_DESCRIPTION - lowest level grouping of customers with similar business product offerings and methods of selling within a trade channel
#     DISTROBUTION_MODE_DSCRIPTION - mode of distribution

customer = pd.read_csv("customer.csv")

# Should have 6334 rows and 7 cols

print(customer.shape)

In [0]:
# customer head
print(customer.head())

In [0]:
# Customer Data Summary Stats
print(customer.info())
print(customer.describe(include="all"))

In [0]:
# missing values
print(customer.isna().sum())

In [0]:
# look at the 4 rows with missing data. Random forest 
print(customer[customer['DISTRIBUTION_MODE_DESCRIPTION'].isna()])

In [0]:
# replace missing values in DISTRIBUTION_MODE_DESCRIPTION with mode.
customer['DISTRIBUTION_MODE_DESCRIPTION'] = customer['DISTRIBUTION_MODE_DESCRIPTION'].fillna("unspecified")
customer['DISTRIBUTION_MODE_DESCRIPTION'].isna().sum()

In [0]:
# sales office distribution
customer['SALES_OFFICE_DESCRIPTION'].value_counts()

In [0]:
# Customer dummy variables:
customer_dummies = pd.get_dummies(customer, columns=['DISTRIBUTION_MODE_DESCRIPTION', 'SHIPPING_CONDITIONS_DESCRIPTION', 'COLD_DRINK_CHANNEL_DESCRIPTION'], drop_first=True)
customer_dummies.head()

In [0]:
# Sales Office distribution
counts1 = customer['SALES_OFFICE_DESCRIPTION'].value_counts()

# Plot
counts1.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Clients per Sales Office")
plt.ylabel("Count")
plt.xlabel("Sales Office")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Cold Drink Channel Distribution
counts1 = customer['COLD_DRINK_CHANNEL_DESCRIPTION'].value_counts()

# Plot
counts1.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Cold Drink Channel Distribution")
plt.ylabel("Count")
plt.xlabel("Cold Drink Channel")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Restaurants are by far the most popular channel. I would be interested to pair this with the material table to see what specific items they order.

In [0]:
# Shipping Conditions Distribution
counts1 = customer['SHIPPING_CONDITIONS_DESCRIPTION'].value_counts()

# Plot
counts1.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Shipping Condition Counts")
plt.ylabel("Count")
plt.xlabel("Shipping Conditions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Shipping conditions are mostly 48 hours. 

### Material

In [0]:
# Loading Material Data

# Specific product info
#     MATERIAL_ID - unique identifier for the product
#     BEV_CAT_DESC - category of drink(juice, energy drink, soda)
#     PACK_TYPE_DESC - package type
#     TRADE_MARK_DESC - brank of product
#     FLAVOUR_DESC - flavor of product
#     PACK_SIZE_DESC - size of product

material = pd.read_csv("material.csv")

# Should have 1252 rows and 6 cols

print(material.shape)

In [0]:
# head
print(material.head())

In [0]:
# Missing values
print(material.isna().sum())

In [0]:
# Material Summary stats
print(material.info())
print(material.describe(include="all"))

In [0]:
# Replace missing values in BEV_CAT_DESC:
empty = material[material['BEV_CAT_DESC'].isna()]
print(empty["TRADE_MARK_DESC"].value_counts())

In [0]:
material['BEV_CAT_DESC'] = material['BEV_CAT_DESC'].fillna('SUNNY_SIP')
material['BEV_CAT_DESC'].isna().sum()

In [0]:
# Material dummy variables:
material_dummies = pd.get_dummies(material, columns=['BEV_CAT_DESC', 'TRADE_MARK_DESC', 'PACK_TYPE_DESC', 'PACK_SIZE_DESC'], drop_first=True)
material_dummies.head()

In [0]:
# Trade Mark Distribution
counts1 = material['TRADE_MARK_DESC'].value_counts()

# Plot
counts1.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Trademark Category Counts")
plt.ylabel("Count")
plt.xlabel("Trade Mark")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Beverage Category Distribution
counts1 = material['BEV_CAT_DESC'].value_counts()

# Plot
counts1.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Beverage Category Counts")
plt.ylabel("Count")
plt.xlabel("Beverage Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Operating Hours

In [0]:
# Loading Operating Hours Data

# Day of the week and how often a customer is scheduled to place an order
#     CUSTOMER_NUMBER - unique identifier for each customer
#     FREQUENCY - how often a customer will receive deliveries
#     DELIVERY_ANCHOR_dAY - day of the week customer receives their order
#     CALLING_ANCHOR_DATE - the day the delivery policy starts

op_hours = pd.read_csv("operating_hours.csv")

# Should have 6202 rows and 4 cols

print(op_hours.shape)

In [0]:
print(op_hours.head())

In [0]:
# Missing values
print(op_hours.isna().sum())

In [0]:
# Operating Hours Summary stats
print(op_hours.info())
print(op_hours.describe(include="all"))

In [0]:
# Convert CALLING_ANCHOR_DATE to datetime
op_hours['CALLING_ANCHOR_DATE'] = pd.to_datetime(op_hours['CALLING_ANCHOR_DATE'], format='%m/%d/%Y')
op_hours['CALLING_ANCHOR_DATE'].dtype

In [0]:
# Operating Hours dummy variables:
ophrs_dummies = pd.get_dummies(op_hours, columns=['FREQUENCY', 'DELIVERY_ANCHOR_DAY'], drop_first=True)
ophrs_dummies.head()

In [0]:
op_hours['DELIVERY_ANCHOR_DAY'] = op_hours['DELIVERY_ANCHOR_DAY'].replace('Error', 'Saturday')

In [0]:
# Delivery Anchor Day Distribution
counts1 = op_hours['DELIVERY_ANCHOR_DAY'].value_counts()

# Plot
counts1.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Delivery Day Counts")
plt.ylabel("Count")
plt.xlabel("Delivery Anchor Day")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Sales

In [0]:
# Loading Sales Data

# Which items were sold
#     CUSTOMER_ID - unique customer identifier
#     POSTING_DATE - date financial transaction if recorded
#     MATERIAL_ID - unique identifier for the product
#     GROSS_PROFIT_DEAD_NET - bottler gross profit after invoice allowance with customer marketing expenses and rebates subtracted
#     NSI_DEAD_NET - bottler sales revnue
#     PHYSICAL_VOLUME - case volume of product in that order

sales = pd.read_csv("sales.csv")

# Should have 499,787 rows and 8 cols

print(sales.shape)

In [0]:
# sales head
print(sales.head())

In [0]:
# sales missing values
print(sales.isna().sum())

In [0]:
# Sales Summary stats
print(sales.info())
print(sales.describe(include="all"))

In [0]:
# Sales Correlations:
sales.corr(numeric_only=True)
# remove customer_id, nsi, and gross_profit

In [0]:
# Convert POSTING_DATE to datetime
sales['POSTING_DATE'] = pd.to_datetime(sales['POSTING_DATE'], format='%Y-%m-%d')
sales['POSTING_DATE'].dtypes

In [0]:
import seaborn as sns
# Select numeric columns (excluding ID-like fields)
numeric_cols = ['GROSS_PROFIT_DEAD_NET', 'PHYSICAL_VOLUME', 'NSI', 'NSI_DEAD_NET', 'GROSS_PROFIT']

# Create boxplots
plt.figure(figsize=(12, 6))
sns.boxplot(data=sales[numeric_cols], orient="h")
plt.title("Overall Distribution of Numeric Values")
plt.show()
# this plot with grubs test to remove proportional outliers

### Orders

In [0]:
# Loading Orders Data

# Items ordered
#     CUSTOMER_ID - unique customer identifier
#     CREATED_DATE_EST - date the order was created
#     CREATED_DATE_UTC - timestamp for order
#     MATERIAL_ID - unique identifier for the product
#     ORDER_QUANTITY - quantity of items
#     ORDER_TYPE - method used by customer to place order
#     PLANT_ID - bottling plant customer receives order from

orders = pd.read_csv("orders.csv")

# Should have 1,662,157 rows and 7 cols

print(orders.shape)

In [0]:
# orders head
print(orders.head())

In [0]:
# orders missing values
print(orders.isna().sum())

In [0]:
orders = orders.dropna()
orders.shape

In [0]:
# Orders Summary stats
print(orders.info())
print(orders.describe(include="all"))

In [0]:
# Convert CREATED_DATE_EST and CREATED_DATE_UTC to datetime
orders['CREATED_DATE_EST'] = pd.to_datetime(orders['CREATED_DATE_EST'], format='%Y-%m-%d')
orders['CREATED_DATE_UTC'] = pd.to_datetime(orders['CREATED_DATE_UTC'])
orders[['CREATED_DATE_UTC', 'CREATED_DATE_EST']].dtypes

In [0]:
# Dropping rows with missing material_id:
orders = orders.dropna(subset=['MATERIAL_ID'])
orders.shape

In [0]:
# convert material id to int:
orders['MATERIAL_ID'] = orders['MATERIAL_ID'].astype('Int64')
orders['MATERIAL_ID'].head()

In [0]:
orders_dummies = pd.get_dummies(orders, columns=['ORDER_TYPE', 'PLANT_ID'], drop_first=True)

In [0]:
#distribution by order type
plt.figure(figsize=(8, 5))
sns.boxplot(x='ORDER_TYPE', y='ORDER_QUANTITY', data=orders)
plt.title('Order Quantity by Order Type')
plt.xticks(rotation=45)
plt.show()

### Visit Plan

In [0]:
# Loading Visit Plan Data

# Day of the week and how often a customer is scheduled to place an order
#     CUSTOMER_ID - unique customer identifier
#     FREQUENCY - how often a customer will receive a delivery
#     ELT_TS - timestamp of when the policy was logged
#     SNAPSHOT_DATE - date of when the policy was logged
#     ANCHOR_DATE - day of the week that customer receives their order
#     SALES_OFFICE - id of sales office the customer orders from
#     SALES_OFFICE_DESC - city of which sales office the customer orders from
#     DISTROBUTION_MODE - mode of distrobution for a customer
#     SHIPPING_CONDITIONS_DESC - general shipping strategy for the delivery of goods from vendor to customer. 

visit_plan = pd.read_csv("visit_plan.csv", low_memory = False)

# Should have 14,796,017 rows and 9 cols

print(visit_plan.shape)

In [0]:
# visit plan head
print(visit_plan.head())

In [0]:
# visit plan missing values
print(visit_plan.isna().sum())

In [0]:
# Dropping rows with empty.
visit_plan = visit_plan.dropna()
visit_plan.shape

In [0]:
# Visit Plan Summary stats
print(visit_plan.info())
print(visit_plan.describe(include="all"))

In [0]:
#converting ELT_TS, ANCHOR_DATE, and SNAPSHOT_DATE to datetime
visit_plan['ELT_TS'] = pd.to_datetime(visit_plan['ELT_TS'])
visit_plan['ANCHOR_DATE'] = pd.to_datetime(visit_plan['ANCHOR_DATE'], format='%Y-%m-%d', errors='coerce')
visit_plan['SNAPSHOT_DATE'] = pd.to_datetime(visit_plan['SNAPSHOT_DATE'], format='%Y-%m-%d', errors='coerce')
visit_plan[['ELT_TS', 'SNAPSHOT_DATE', 'ANCHOR_DATE']].dtypes

In [0]:
# converting frequency to int:
# finding unique sales offices
unique_freq = visit_plan["FREQUENCY"].unique()
print(unique_freq)

# no idea what any of this means

In [0]:
# Distribution mode chart
counts = visit_plan['DISTRIBUTION_MODE'].value_counts()

# Plot
counts.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Frequency of Distribution Modes")
plt.ylabel("Count")
plt.xlabel("Distribution Mode")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Shipping Conditions chart
counts = visit_plan['SHIPPING_CONDITIONS_DESC'].value_counts()

# Plot
counts.plot(kind='bar', figsize=(8,5), color='skyblue')
plt.title("Shipping Condition Counts")
plt.ylabel("Count")
plt.xlabel("Shipping Conditions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# creating dummies for visit plan
visit_plan_dummies = pd.get_dummies(visit_plan, columns=['SHIPPING_CONDITIONS_DESC', 'DISTRIBUTION_MODE', 'SALES_OFFICE_DESC', 'SALES_OFFICE', 'FREQUENCY'], drop_first=True)

In [0]:
# sample anchor dates for random customers
visit_plan.loc[visit_plan["CUSTOMER_ID"] == 500431853].groupby("ANCHOR_DATE").count()

In [0]:
visit_plan.loc[visit_plan["CUSTOMER_ID"] == 600076074].groupby("ANCHOR_DATE").count()

### Google Analytics

In [0]:
# Loading Google Analytics Data

# Website events
#     CUSTOMER_ID - unique identifier for each customer
#     EVENT_DATE - date of event on myCoke360
#     EVENT_TIMESTAMP - timestamp of event
#     EVENT_NAME - action the customer took
#     DEVICE_CATEGORY - type of device
#     DEVICE_MOBILE_BRAND_NAME - brand of device
#     DEVICE_OPERATING_SYSTEM - OS of device used
#     EVENT_PAGE_NAME - name of page viewed
#     EVENT_PAGE_TITLE - title of page viewed
#     ITEMS - list of material ids and quantities

google_a = pd.read_csv("google_analytics.csv")

# should have 3,704,088 rows and 10 cols

print(google_a.shape)

In [0]:
# google head
print(google_a.head())

In [0]:
# google missing values
print(google_a.isna().sum())

In [0]:
# Google Analytics Summary stats
print(google_a.info())
print(google_a.describe(include="all"))

In [0]:
# replacing missing values
google_a['DEVICE_MOBILE_BRAND_NAME'] = google_a['DEVICE_MOBILE_BRAND_NAME'].fillna('OTHER')
google_a['EVENT_PAGE_NAME'] = google_a['EVENT_PAGE_NAME'].fillna('UNKNOWN_PAGE')
google_a['EVENT_PAGE_TITLE'] = google_a['EVENT_PAGE_TITLE'].fillna('UNKNOWN_TITLE')
google_a.isna().sum()

In [0]:
# changing EVENT_DATE and EVENT_TIMESTAMP
google_a['EVENT_TIMESTAMP'] = pd.to_datetime(google_a['EVENT_TIMESTAMP'])
google_a['EVENT_DATE'] = pd.to_datetime(google_a['EVENT_DATE'], format='%Y-%m-%d')
google_a[['EVENT_TIMESTAMP', 'EVENT_DATE']].dtypes

In [0]:
# exploration
google_a['EVENT_NAME'].unique()

In [0]:
cart_abandonment_events = [
    # Core funnel
    "add_to_cart",
    "update_cart",
    "remove_from_cart",
    "view_cart",
    "begin_checkout",
    "proceed_to_checkout",
    "CheckoutPage_Displayed",
    "add_payment_info",
    "add_shipping_info",
    "OrderSubmit_CheckoutPage_Failed",

    # Cart interactions
    "UpdateCart_Cart_Retrieved",
    "UpdateCart_Cart_Clicked",
    "CartPage_Displayed",
    "CartProductQuantity_Cart_Changed",
    "ProductCheckmark_Cart_Checked",
    "ProductCheckmark_Cart_Unchecked",
    "SelectAll_Cart_Checked",
    "SelectAll_Cart_Unchecked",
    "ContinueShopping_Cart_Clicked",
    "export_cart_click",

    # Errors / failures
    "Error_post_create_webcart",
    "Error_Get_cart_data",
    "Update_Cart_Item_With_Price_Data_Failed",
    "Update_Cart_With_Details_Failed",
    "Update_Cart_Details_For_Payment_Failed",
    "CheckoutData_Retrieve_Failed",
    "On_Proceed_To_Checkout_Click_Failed",
    "Update_Web_Cart_Failed",
    "Error_get_webcart_details",
    "Get_Active_Cart_Items_Failed"]

important_events = ["add_to_cart", "proceed_to_checkout", "remove_from_cart", "update_cart", "begin_checkout"]

In [0]:
# Cart abandonment items by device category
google_a.loc[google_a['EVENT_NAME'].isin(cart_abandonment_events)].groupby(['DEVICE_CATEGORY', 'EVENT_NAME'])['EVENT_NAME'].count().reset_index(name='event_count').sort_values(by='event_count', ascending=False)

In [0]:
table = (
    google_a[google_a['EVENT_NAME'].isin(important_events)]
    .groupby(['DEVICE_CATEGORY', 'EVENT_NAME'])['EVENT_NAME'].count().reset_index(name='event_count').pivot(index='DEVICE_CATEGORY', columns='EVENT_NAME', values='event_count')
    .fillna(0)
)

table.plot(kind='bar', figsize=(10,6))
plt.title('Top 5 Cart Abandonment Events by Device Category')
plt.xlabel('Device Category')
plt.ylabel('Event Count')
plt.xticks(rotation=0)
plt.legend(title='Event Name', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
ga_dummies = pd.get_dummies(google_a, columns=['EVENT_NAME', 'DEVICE_CATEGORY', 'DEVICE_MOBILE_BRAND_NAME', 'DEVICE_OPERATING_SYSTEM', 'EVENT_PAGE_NAME'], drop_first=True)

In [0]:
# Extract weekday (0 = Monday, 6 = Sunday)
google_a['WEEKDAY'] = google_a['EVENT_DATE'].dt.day_name()

# Focus only on the cart abandonment events
cart_abandonment_events = ["add_to_cart", "proceed_to_checkout", "remove_from_cart", "update_cart", "begin_checkout"]

# Filter to those events
filtered = google_a[google_a['EVENT_NAME'].isin(cart_abandonment_events)]

# Count by weekday and event
event_counts = (
    filtered
    .groupby(['WEEKDAY','EVENT_NAME'])
    .size()
    .reset_index(name='count')
)

# Pivot for visualization (event types as columns, weekdays as rows)
pivot = event_counts.pivot(index='WEEKDAY', columns='EVENT_NAME', values='count').fillna(0)

print(pivot)

# Plot (stacked bar so you see total volume + breakdown per day)
pivot.plot(kind='bar', stacked=True, figsize=(14,6))
plt.title("Cart Abandonment Events by Day of Week")
plt.ylabel("Count")
plt.xlabel("Day of Week")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Joining Tables



# -----------------------------
# 4) Use sales as the base table

# 5) Join all customer-key tables except material
# -----------------------------
customer_key_tables = [
    "google_a", "orders", "customer",
    "cutoff", "op_hours", "visit_plan"
]

for tname in customer_key_tables:
    right = sales[tname]
    if "customer_key" not in right.columns:
        continue
    right_pruned = drop_dupe_cols_on_join(base, right, join_keys=["customer_key"])
    base = base.merge(right_pruned, on="customer_key", how="left")

# -----------------------------
# 6) Join material on material_id (or best available key)
# -----------------------------
material_df = dfs["material"].copy()
if "material_id" not in material_df.columns:
    mk = best_material_key(material_df)
    if mk and mk != "material_id":
        material_df = material_df.rename(columns={mk: "material_id"})

base_mat_key = best_material_key(base)
if base_mat_key:
    material_pruned = drop_dupe_cols_on_join(base, material_df, join_keys=["material_id"])
    base = base.merge(material_pruned, left_on=base_mat_key, right_on="material_id", how="left")
    if "material_id_y" in base.columns:  # drop duplicate id from right
        base = base.drop(columns=["material_id_y"]).rename(columns={"material_id_x": "material_id"})
else:
    base["material_id"] = np.nan

# -----------------------------
# 7) Final joined DataFrame
# -----------------------------
joined_df = base

print(joined_df.info())
print(joined_df.head(10))